# Step 3: Generate Dataset with Qwen3-32B via vLLM

For each FAERS record, Qwen3-32B fills in only the fields that cannot be derived from structured data:

| Field | Source | Reliability |
|-------|--------|-------------|
| `narrative` | **Qwen3-32B generated** (all records) | Good — consistent with ground truth labels |
| `seriousness`, `criteria` | FAERS (ground truth) | ✅ High |
| `meddra_pt`, `meddra_soc` | MedDRA lookup (notebook 02) | ✅ High |
| `who_umc_category` | FAERS fields — deterministic (notebook 01) | ✅ Medium |
| `labelling_status` | **FDA drug label API + Qwen3-32B** | ✅ Medium-High |
| ~~`naranjo_score`~~ | Dropped — unreliable from synthetic narrative | — |

## Reliability improvements vs. original design
- **WHO-UMC**: computed from FAERS `drugadditional`/`drugrecurrence`/`outcome` fields — no LLM
- **Labelling status**: Qwen3-32B receives the actual FDA prescribing information adverse reactions
  section as context, rather than relying on parametric memory alone

## Prerequisites
```bash
vllm serve Qwen/Qwen3-32B --dtype bfloat16 --max-model-len 4096 --port 8000
```

## Checkpointing
Results are saved every 200 records. Re-running the notebook resumes from the last checkpoint.

In [ ]:
%pip install openai httpx tqdm -q

In [ ]:
import json
import asyncio
import time
import re
from pathlib import Path
from tqdm.notebook import tqdm
from openai import AsyncOpenAI
import httpx

DATA_DIR = Path("data")

FAERS_FILE        = DATA_DIR / "faers_raw.json"
LOOKUP_FILE       = DATA_DIR / "meddra_lookup.json"
LABEL_CACHE_FILE  = DATA_DIR / "drug_labels.json"   # cached FDA label ADR sections
OUTPUT_FILE       = DATA_DIR / "dataset_raw.json"

VLLM_BASE_URL = "http://localhost:8000/v1"
MODEL_NAME    = "Qwen/Qwen3-32B"
FDA_LABEL_URL = "https://api.fda.gov/drug/label.json"

CONCURRENCY      = 8
CHECKPOINT_EVERY = 200

print("Config loaded.")

In [ ]:
# ── Load input data ──────────────────────────────────────────────────────────
with open(FAERS_FILE) as f:
    faers_records = json.load(f)

with open(LOOKUP_FILE) as f:
    pt_lookup = json.load(f)

print(f"FAERS records: {len(faers_records)}")
print(f"MedDRA lookup entries: {len(pt_lookup)}")

In [ ]:
# ── vLLM health check ────────────────────────────────────────────────────────
import httpx

try:
    resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=5)
    models = resp.json()
    print(f"✓ vLLM is running. Available models:")
    for m in models.get("data", []):
        print(f"  - {m['id']}")
except Exception as e:
    print(f"✗ vLLM not reachable: {e}")
    print("Start vLLM first:")
    print("  vllm serve Qwen/Qwen3-32B --dtype bfloat16 --max-model-len 4096 --port 8000")

In [ ]:
# ── MedDRA fuzzy lookup (same as notebook 02) ────────────────────────────────
from rapidfuzz import process, fuzz

ALL_PT_KEYS = list(pt_lookup.keys())

def lookup_pt(pt_name: str, threshold: int = 80) -> dict:
    """Return MedDRA hierarchy dict for a PT name. Returns empty dict if not found."""
    key = pt_name.lower().strip()
    if key in pt_lookup:
        return pt_lookup[key]
    result = process.extractOne(key, ALL_PT_KEYS, scorer=fuzz.WRatio, score_cutoff=threshold)
    if result:
        return pt_lookup[result[0]]
    return {}

print("lookup_pt ready.")

In [ ]:
# ── Prompt templates ─────────────────────────────────────────────────────────

NARRATIVE_PROMPT = """You are a pharmacovigilance medical writer. Write a realistic, concise clinical safety narrative (3-6 sentences) based on this adverse event report.

Report data:
- Drug: {drug_name}{dose_info}
- Adverse event: {reaction_pt}
- Indication: {indication}
- Patient: {age_sex}
- Outcome: {outcome}
- Serious: {serious}

Write the narrative in third person. Include temporal relationship between drug and event. Keep it factual and concise. Return ONLY the narrative text, no labels or JSON."""


# Used when we have actual FDA label text — primary path
LABELLING_PROMPT_WITH_LABEL = """You are a senior pharmacovigilance specialist assessing labelling status.

Drug: {drug_name}
Adverse Event (MedDRA PT): {reaction_pt}

Adverse Reactions section from the official FDA prescribing information:
---
{label_excerpt}
---

Is "{reaction_pt}" an Expected or Unexpected adverse reaction based on this label?
- Expected: this reaction (or a closely related term) is mentioned in the adverse reactions section
- Unexpected: this reaction is not listed in the label

Return ONLY valid JSON with exactly these two keys:
{{
  "labelling_status": "Expected" or "Unexpected",
  "labelling_evidence": "<one sentence citing the specific label text or explaining why it is not listed>"
}}"""


# Fallback when no FDA label was found for the drug
LABELLING_PROMPT_NO_LABEL = """You are a senior pharmacovigilance specialist assessing labelling status.

Drug: {drug_name}
Adverse Event (MedDRA PT): {reaction_pt}
Indication: {indication}

Based on your medical knowledge of {drug_name} and its drug class, is "{reaction_pt}" a known adverse reaction typically listed in this drug's prescribing information?
- Expected: the reaction is commonly associated with this drug or its class and would appear in the adverse reactions / warnings section
- Unexpected: the reaction is not typically associated with this drug or its class

Return ONLY valid JSON with exactly these two keys:
{{
  "labelling_status": "Expected" or "Unexpected",
  "labelling_evidence": "<one sentence explaining the pharmacological basis or why this reaction is not expected>"
}}"""


def extract_json(text: str) -> dict | None:
    """Extract JSON from model output. Handles markdown code blocks and leading text."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"({.*})", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return None


print("Prompts and helpers defined.")

In [ ]:
# ── Async generation function ─────────────────────────────────────────────────

client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")

async def generate_text(prompt: str, max_tokens: int = 300, temperature: float = 0.4) -> str:
    """Single async call to vLLM."""
    response = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return response.choices[0].message.content.strip()


async def process_record(record: dict) -> dict | None:
    """Process one FAERS record into a complete training example.

    LLM calls per record:
      1. Narrative generation (all records — FAERS has no narratives)
      2. Labelling assessment (with FDA label context when available)

    WHO-UMC comes from FAERS structured fields (deterministic, no LLM).
    Naranjo is dropped entirely.
    """
    try:
        drug     = record["drug_name"]
        reaction = record["reaction_pt"]

        # ── Step 1: MedDRA hierarchy ───────────────────────────────────────────
        meddra = lookup_pt(reaction)
        if not meddra:
            return None

        # ── Step 2: Generate narrative ─────────────────────────────────────────
        age        = record.get("patient_age", "")
        sex        = record.get("patient_sex", "unknown")
        dose       = record.get("drug_dose", "")
        indication = record.get("drug_indication", "") or "unknown indication"
        outcome    = record.get("reaction_outcome", "Unknown")
        serious    = "Serious" if record.get("serious") else "Non-serious"

        narrative_prompt = NARRATIVE_PROMPT.format(
            drug_name   = drug,
            dose_info   = f" {dose}" if dose else "",
            reaction_pt = reaction,
            indication  = indication,
            age_sex     = f"{age}-year-old {sex}" if age else sex,
            outcome     = outcome,
            serious     = serious,
        )
        narrative = await generate_text(narrative_prompt, max_tokens=250, temperature=0.5)

        # ── Step 3: Labelling status — with FDA label when available ──────────
        label_text = drug_labels.get(drug, "")

        if label_text:
            labelling_prompt = LABELLING_PROMPT_WITH_LABEL.format(
                drug_name     = drug,
                reaction_pt   = reaction,
                label_excerpt = label_text,
            )
        else:
            labelling_prompt = LABELLING_PROMPT_NO_LABEL.format(
                drug_name   = drug,
                reaction_pt = reaction,
                indication  = indication,
            )

        labelling_text = await generate_text(labelling_prompt, max_tokens=200, temperature=0.1)
        assessment = extract_json(labelling_text)

        if not assessment or "labelling_status" not in assessment:
            return None

        # Validate labelling_status is one of the two expected values
        if assessment["labelling_status"] not in ("Expected", "Unexpected"):
            return None

        # ── Step 4: WHO-UMC from FAERS — no LLM needed ────────────────────────
        # Computed deterministically in notebook 01 from drugadditional/drugrecurrence/outcome
        who_umc = record.get("who_umc_preliminary", "Unassessable")

        # ── Assemble final record ──────────────────────────────────────────────
        return {
            "id":               record["id"],
            "narrative":        narrative,
            "narrative_source": "generated",
            # Seriousness — ground truth from FAERS
            "seriousness":           serious,
            "seriousness_criteria":  record.get("seriousness_criteria", []),
            # MedDRA — ground truth from lookup
            "meddra_llt":       meddra.get("meddra_llt",      meddra["meddra_pt"]),
            "meddra_llt_code":  meddra.get("meddra_llt_code"),
            "meddra_pt":        meddra["meddra_pt"],
            "meddra_pt_code":   meddra.get("meddra_pt_code"),
            "meddra_soc":       meddra.get("meddra_soc", ""),
            "meddra_soc_code":  meddra.get("meddra_soc_code"),
            # Labelling — Qwen3-32B with FDA label context
            "labelling_status":      assessment.get("labelling_status"),
            "labelling_evidence":    assessment.get("labelling_evidence", ""),
            "label_source":          "fda_label" if label_text else "llm_knowledge",
            # WHO-UMC — deterministic from FAERS fields
            "who_umc_category":      who_umc,
        }

    except Exception:
        return None


print("Async functions defined.")

In [ ]:
# ── Async generation function ─────────────────────────────────────────────────

client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")

async def generate_text(prompt: str, max_tokens: int = 300, temperature: float = 0.4) -> str:
    """Single async call to vLLM."""
    response = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # Disable thinking mode
    )
    return response.choices[0].message.content.strip()


async def process_record(record: dict) -> dict | None:
    """Process one FAERS record into a complete training example."""
    try:
        drug = record["drug_name"]
        reaction = record["reaction_pt"]

        # ── Step 1: Get MedDRA hierarchy ──────────────────────────────────────
        meddra = lookup_pt(reaction)
        if not meddra:
            return None  # Skip records we can't map

        # ── Step 2: Generate narrative if missing ─────────────────────────────
        narrative = record.get("narrative", "").strip()
        if not narrative or len(narrative) < 50:
            age = record.get("patient_age", "")
            sex = record.get("patient_sex", "unknown")
            dose = record.get("drug_dose", "")
            indication = record.get("drug_indication", "unknown indication")
            outcome = record.get("reaction_outcome", "Unknown")
            serious = "Serious" if record.get("serious") else "Non-serious"

            age_sex = f"{age}-year-old {sex}" if age else sex
            dose_info = f" {dose}" if dose else ""

            narrative_prompt = NARRATIVE_PROMPT.format(
                drug_name=drug,
                dose_info=dose_info,
                reaction_pt=reaction,
                indication=indication or "unknown indication",
                age_sex=age_sex,
                outcome=outcome,
                serious=serious,
            )
            narrative = await generate_text(narrative_prompt, max_tokens=250, temperature=0.5)

        # ── Step 3: Generate causality and labelling ──────────────────────────
        serious_label = "Serious" if record.get("serious") else "Non-serious"
        outcome = record.get("reaction_outcome", "Unknown")

        assessment_prompt = ASSESSMENT_PROMPT.format(
            drug_name=drug,
            reaction_pt=reaction,
            serious=serious_label,
            outcome=outcome,
            narrative=narrative[:800],  # truncate very long narratives in prompt
        )
        assessment_text = await generate_text(assessment_prompt, max_tokens=300, temperature=0.2)
        assessment = extract_json(assessment_text)

        if not assessment:
            return None  # Skip if JSON extraction failed

        # Validate required keys are present
        required_keys = ["naranjo_score", "naranjo_category", "who_umc_category",
                         "labelling_status", "labelling_evidence"]
        if not all(k in assessment for k in required_keys):
            return None

        # ── Assemble final record ─────────────────────────────────────────────
        return {
            "id": record["id"],
            "narrative": narrative,
            "narrative_source": "faers" if record.get("has_narrative") else "generated",
            # Seriousness (from FAERS)
            "seriousness": serious_label,
            "seriousness_criteria": record.get("seriousness_criteria", []),
            # MedDRA (from lookup)
            "meddra_llt":      meddra.get("meddra_llt", meddra["meddra_pt"]),
            "meddra_llt_code": meddra.get("meddra_llt_code"),
            "meddra_pt":       meddra["meddra_pt"],
            "meddra_pt_code":  meddra.get("meddra_pt_code"),
            "meddra_soc":      meddra.get("meddra_soc", ""),
            "meddra_soc_code": meddra.get("meddra_soc_code"),
            # Causality + labelling (from Qwen3-32B)
            "naranjo_score":      int(assessment.get("naranjo_score", 0)),
            "naranjo_category":   assessment.get("naranjo_category", "Unknown"),
            "who_umc_category":   assessment.get("who_umc_category", "Unknown"),
            "labelling_status":   assessment.get("labelling_status", "Unknown"),
            "labelling_evidence": assessment.get("labelling_evidence", ""),
        }

    except Exception as e:
        # Don't crash the whole loop; skip bad records silently
        return None


print("Async functions defined.")

In [ ]:
# ── Quick single-record test before full run ─────────────────────────────────
test_record = faers_records[0]
print(f"Testing with record: drug={test_record['drug_name']}, reaction={test_record['reaction_pt']}")

result = await process_record(test_record)

if result:
    print("\n✓ Test passed. Sample output:")
    print(json.dumps(result, indent=2))
else:
    print("✗ Test failed. Check vLLM server and FAERS data.")

In [ ]:
# ── Load existing checkpoint ──────────────────────────────────────────────────
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        completed = json.load(f)
    completed_ids = {r["id"] for r in completed}
    print(f"Resuming from checkpoint: {len(completed)} records already done.")
else:
    completed = []
    completed_ids = set()
    print("Starting fresh generation.")

# Records still to process
todo = [r for r in faers_records if r["id"] not in completed_ids]
print(f"Records remaining: {len(todo)}")

In [ ]:
# ── Quality check ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
print(f"Total records: {len(df)}")

print(f"\nSeriousness distribution:")
print(df["seriousness"].value_counts())

print(f"\nLabelling status:")
print(df["labelling_status"].value_counts())

print(f"\nLabel source (fda_label vs llm_knowledge):")
print(df["label_source"].value_counts())
print(f"  → {(df['label_source']=='fda_label').mean():.1%} of records used actual FDA label text")

print(f"\nWHO-UMC category (from FAERS — deterministic):")
print(df["who_umc_category"].value_counts())

print(f"\nTop 10 SOCs:")
print(df["meddra_soc"].value_counts().head(10))

print(f"\nNarrative length (chars):")
print(df["narrative"].str.len().describe().round(0))

In [ ]:
# ── Quality check ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
print(f"Total records: {len(df)}")
print(f"\nNarrative source:")
print(df["narrative_source"].value_counts())
print(f"\nSeriousness distribution:")
print(df["seriousness"].value_counts())
print(f"\nLabelling status:")
print(df["labelling_status"].value_counts())
print(f"\nNaranjo category:")
print(df["naranjo_category"].value_counts())
print(f"\nWHO-UMC category:")
print(df["who_umc_category"].value_counts())
print(f"\nNaranjo score distribution:")
print(df["naranjo_score"].describe())

# Check SOC coverage
print(f"\nTop 10 SOCs:")
print(df["meddra_soc"].value_counts().head(10))

## Done
Raw generated dataset saved to `data/dataset_raw.json`. Proceed to `04_format_dataset.ipynb`.